In [1]:
import os
import shutil
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, DateType
from pyspark.sql import Row
from pyspark.sql.functions import expr
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Managing Nulls")
    .config("spark.master", "local[*]")
    .getOrCreate()
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/21 16:36:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/21 16:36:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/21 16:36:55 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
moviesDF = spark.read.json("src/main/resources/data/movies.json")

moviesDF.show(5)

+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|       Creative_Type|Director|Distributor|IMDB_Rating|IMDB_Votes|MPAA_Rating|Major_Genre|Production_Budget|Release_Date|Rotten_Tomatoes_Rating|Running_Time_min|             Source|               Title|US_DVD_Sales|US_Gross|Worldwide_Gross|
+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|                NULL|    NULL|   Gramercy|        6.1|      1071|          R|       NULL|          8000000|   12-Jun-98|                  NULL|            NULL|               NULL|      The Land Girls|        NULL|  146083|         146083|
|                NULL|    NULL|     

In [ ]:
# get first non null - for each movie title use either imdb or rotten tomatoes rating, whichever is not null

nonNullRatings = (
    moviesDF.select(
        col("Title"),
        col("Rotten_Tomatoes_Rating"),
        col("IMDB_Rating"),
        coalesce(col("Rotten_Tomatoes_Rating"), col("IMDB_Rating") * 10)
    )
)


nonNullRatings.show(5)

+--------------------+----------------------+-----------+----------------------------------------------------+
|               Title|Rotten_Tomatoes_Rating|IMDB_Rating|coalesce(Rotten_Tomatoes_Rating, (IMDB_Rating * 10))|
+--------------------+----------------------+-----------+----------------------------------------------------+
|      The Land Girls|                  NULL|        6.1|                                                61.0|
|First Love, Last ...|                  NULL|        6.9|                                                69.0|
|I Married a Stran...|                  NULL|        6.8|                                                68.0|
|Let's Talk About Sex|                    13|       NULL|                                                13.0|
|                Slam|                    62|        3.4|                                                62.0|
+--------------------+----------------------+-----------+----------------------------------------------------+
o

In [5]:
# checking for nulls
nullCount = (
    moviesDF.select("*")
    .where(col("Rotten_Tomatoes_Rating").isNull())
)

nullCount.show(5)

+-------------+-----------------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+------+--------------------+------------+--------+---------------+
|Creative_Type|         Director|Distributor|IMDB_Rating|IMDB_Votes|MPAA_Rating|Major_Genre|Production_Budget|Release_Date|Rotten_Tomatoes_Rating|Running_Time_min|Source|               Title|US_DVD_Sales|US_Gross|Worldwide_Gross|
+-------------+-----------------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+------+--------------------+------------+--------+---------------+
|         NULL|             NULL|   Gramercy|        6.1|      1071|          R|       NULL|          8000000|   12-Jun-98|                  NULL|            NULL|  NULL|      The Land Girls|        NULL|  146083|         146083|
|         NULL|             NULL|     Strand|        6.9|       207|          R|

In [ ]:
# nulls when ordering
# do we put nulls first or last when sorting

(
    moviesDF.orderBy(col("IMDB_Rating").desc_nulls_last())
)

In [6]:
# removing nulls
(
    moviesDF.select("Title", "IMDB_Rating").na.drop() # remove row containing nulls
    .show(5)
)

+--------------------+-----------+
|               Title|IMDB_Rating|
+--------------------+-----------+
|      The Land Girls|        6.1|
|First Love, Last ...|        6.9|
|I Married a Stran...|        6.8|
|                Slam|        3.4|
|           Following|        7.7|
+--------------------+-----------+
only showing top 5 rows



In [ ]:
# replace nulls
moviesDF.na.fill(0, ["IMDB_Rating", "Rotten_Tomatoes_Rating"]).show(5)

+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|       Creative_Type|Director|Distributor|IMDB_Rating|IMDB_Votes|MPAA_Rating|Major_Genre|Production_Budget|Release_Date|Rotten_Tomatoes_Rating|Running_Time_min|             Source|               Title|US_DVD_Sales|US_Gross|Worldwide_Gross|
+--------------------+--------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|                NULL|    NULL|   Gramercy|        6.1|      1071|          R|       NULL|          8000000|   12-Jun-98|                     0|            NULL|               NULL|      The Land Girls|        NULL|  146083|         146083|
|                NULL|    NULL|     

In [ ]:
# fill method is heavily overloaded it even has map where you can map col to default value if null
(
    moviesDF.na.fill({
        "IMDB_Rating": 0,
        "Rotten_Tomatoes_Rating": 10,
        "Director": "Unknown"
    })
    .show(10)
)

+--------------------+-----------------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|       Creative_Type|         Director|Distributor|IMDB_Rating|IMDB_Votes|MPAA_Rating|Major_Genre|Production_Budget|Release_Date|Rotten_Tomatoes_Rating|Running_Time_min|             Source|               Title|US_DVD_Sales|US_Gross|Worldwide_Gross|
+--------------------+-----------------+-----------+-----------+----------+-----------+-----------+-----------------+------------+----------------------+----------------+-------------------+--------------------+------------+--------+---------------+
|                NULL|          Unknown|   Gramercy|        6.1|      1071|          R|       NULL|          8000000|   12-Jun-98|                    10|            NULL|               NULL|      The Land Girls|        NULL|  146083|         146083|


In [ ]:
# complex operations
# ifnull is same as coalese
(
    moviesDF.selectExpr(
        "Title",
        "IMDB_Rating",
        "Rotten_Tomatoes_Rating",
        "ifnull(Rotten_Tomatoes_Rating, IMDB_Rating * 10) as ifnull", # same as coalese
        "nvl(Rotten_Tomatoes_Rating, IMDB_Rating * 10) as nvl", # does same as ifnull
        "nullif(Rotten_Tomatoes_Rating, IMDB_Rating * 10) as nullif", # return null if values are EQUAL, else first value
        "nvl2(Rotten_Tomatoes_rating, IMDB_Rating * 10, 0.0) as nvl2", # if (first != null) second else third 
    )
    .show(5)
)

+--------------------+-----------+----------------------+------+----+------+----+
|               Title|IMDB_Rating|Rotten_Tomatoes_Rating|ifnull| nvl|nullif|nvl2|
+--------------------+-----------+----------------------+------+----+------+----+
|      The Land Girls|        6.1|                  NULL|  61.0|61.0|  NULL| 0.0|
|First Love, Last ...|        6.9|                  NULL|  69.0|69.0|  NULL| 0.0|
|I Married a Stran...|        6.8|                  NULL|  68.0|68.0|  NULL| 0.0|
|Let's Talk About Sex|       NULL|                    13|  13.0|13.0|    13|NULL|
|                Slam|        3.4|                    62|  62.0|62.0|    62|34.0|
+--------------------+-----------+----------------------+------+----+------+----+
only showing top 5 rows

